# 02 - From-scratch DPO on HH-RLHF
Attach the output of notebook 01 as an input dataset.

In [ ]:
# ---- Task 1: repository + environment setup (Kaggle, GPU T4 x2) ----
# Private repo: add a GitHub token as a Kaggle secret named GITHUB_TOKEN
# (Add-ons -> Secrets -> Add secret, then tick it for this notebook).
REPO_URL = "https://github.com/zainabusman22/advanced-genai-pa2.git"
REPO = "/kaggle/working/advanced-genai-pa2"

import os, subprocess, sys
if not os.path.exists(REPO):
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    authed = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
    subprocess.run(["git", "clone", "--depth", "1", authed, REPO], check=True, capture_output=True)
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", REPO_URL], check=True)  # don't keep the token on disk
    del token, authed
os.chdir(REPO)
!git -C {REPO} log --oneline -1
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest
!python -m pip install -q -e . --no-deps
!nvidia-smi

In [ ]:
import torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() == 2, "Select the GPU T4 x2 accelerator"

In [ ]:
# Locate artifacts from earlier notebooks attached as input datasets.
import glob
def find_dir(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"nothing matching {pattern} under /kaggle/input - attach the previous notebook's output")
    print(pattern, "->", hits[0])
    return os.path.dirname(hits[0])

In [ ]:
SFT_BEST = find_dir("sft/best/config.json")
DATA = find_dir("data/processed/hh_train.jsonl")

In [ ]:
!python scripts/train_dpo.py --sft_path {SFT_BEST} \
    --train_path {DATA}/hh_train.jsonl --val_path {DATA}/hh_val.jsonl \
    --output_dir outputs/dpo_hh --beta 0.1 --lr 1e-6 --grad_accum 32

In [ ]:
# Archive outputs so they survive the session (then: Save Version -> attach as dataset).
!cd {REPO} && zip -qr /kaggle/working/dpo_hh_run_outputs.zip outputs/dpo_hh
!ls -lh /kaggle/working/*.zip